💻 노트북 검증용 셀 코드

In [22]:
import torch
import torch.nn as nn
import timm

class FeatureAlignmentBridge(nn.Module):
    def __init__(self, swin_channels, yolo_channels):
        super().__init__()
        # Swin의 출력(C1, C2, C3)을 YOLO의 입력 채널들로 변환
        self.bridge_p3 = nn.Conv2d(swin_channels[0], yolo_channels[0], kernel_size=1)
        self.bridge_p4 = nn.Conv2d(swin_channels[1], yolo_channels[1], kernel_size=1)
        self.bridge_p5 = nn.Conv2d(swin_channels[2], yolo_channels[2], kernel_size=1)
        self.act = nn.SiLU(inplace=True)

    def forward(self, features):
        # features는 [Batch, H, W, Channel] 순서일 수 있으므로 permute 처리
        p3 = self.act(self.bridge_p3(features[0].permute(0, 3, 1, 2)))
        p4 = self.act(self.bridge_p4(features[1].permute(0, 3, 1, 2)))
        p5 = self.act(self.bridge_p5(features[2].permute(0, 3, 1, 2)))
        return [p3, p4, p5]

print("브릿지 레이어 정의 완료")

브릿지 레이어 정의 완료


In [24]:
# 1. 브릿지 동작을 위한 더미 채널 설정 (Swin-Tiny 예시 채널)
# Swin-Tiny의 3단계 출력 채널: [192, 384, 768]
# YOLO Neck의 요구 입력 채널: [256, 512, 1024]
swin_channels = [192, 384, 768]
yolo_channels = [256, 512, 1024]

# 2. 브릿지 인스턴스화
bridge = FeatureAlignmentBridge(swin_channels, yolo_channels)

# 3. Swin 백본에서 나올 법한 더미 피처맵 생성 (Batch=1, H, W, Channel)
features = [
    torch.randn(1, 160, 160, 192), # P3
    torch.randn(1, 80, 80, 384),   # P4
    torch.randn(1, 40, 40, 768)    # P5
]

# 4. 브릿지 통과 테스트
aligned_features = bridge(features)

# 5. 결과 확인 (Print문을 통한 검증)
for i, f in enumerate(aligned_features):
    print(f"변환된 P{i+3} 채널 shape: {f.shape}")

변환된 P3 채널 shape: torch.Size([1, 256, 160, 160])
변환된 P4 채널 shape: torch.Size([1, 512, 80, 80])
변환된 P5 채널 shape: torch.Size([1, 1024, 40, 40])


In [25]:
import sys
import os

# 현재 노트북의 절대 경로를 가져와서, 상위 폴더(프로젝트 루트)를 계산합니다.
current_dir = os.path.abspath('') 
project_root = os.path.dirname(current_dir)

# 계산된 프로젝트 루트를 sys.path의 맨 앞에 추가합니다.
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ 프로젝트 루트 경로 설정 완료: {project_root}")

# 이제 프로젝트 루트에 있는 models 폴더를 정상적으로 인식합니다.
import torch
import torch.nn as nn
import timm
from ultralytics.nn.tasks import DetectionModel

print("✅ 기본 라이브러리 및 Ultralytics 모듈 임포트 성공!")

✅ 프로젝트 루트 경로 설정 완료: c:\Users\tk\Desktop\Swin-YOLO26\swin-yolo26-paint-defect
✅ 기본 라이브러리 및 Ultralytics 모듈 임포트 성공!


💻 [노트북 셀 2: 백본과 YOLO Neck의 채널 불일치 현상 파악]

In [29]:
import torch
from models.swin_yolo26 import SwinYOLO26

# 1. 모델 인스턴스화 (에러가 나기 전 단계까지만 초기화됨)
model = SwinYOLO26(swin_size='n', yolo_size='m')
model.eval() # 평가 모드로 설정

# 2. 더미 입력 데이터 생성 (배치 1, 3채널, 640x640 이미지)
dummy_input = torch.randn(1, 3, 640, 640)

print("--- [1단계] Swin 백본 출력 채널 확인 ---")
# Swin 백본만 단독으로 통과시켜 출력 형태 확인
with torch.no_grad():
    swin_features = model.swin_backbone(dummy_input)

swin_out_channels = []
for i, f in enumerate(swin_features):
    swin_out_channels.append(f.shape[1])
    print(f"Swin P{i+3} 피처맵 Shape: {f.shape} -> 채널 수: {f.shape[1]}")

print("\n--- [2단계] YOLO Neck 레이어 요구사항 분석 ---")
# YOLO 모델의 10번 레이어(Neck 시작점)부터 순회하며 입력/출력 채널 요구사항 확인
# 에러가 났던 Concat 레이어(1536 채널 요구 등)의 정확한 위치를 파악합니다.
for i, m in enumerate(model.model):
    if i >= 10:
        # 모듈 이름과 레이어 번호 출력
        layer_type = type(m).__name__
        
        # Concat 레이어인 경우, 어떤 인덱스(m.f)의 피처맵들을 합치는지 확인
        if layer_type == "Concat":
            print(f"Layer {i} [{layer_type}]: 이전 피처맵 인덱스 {m.f} 를 합칩니다.")
        
        # 입력 채널(c1) 속성이 있는 경우 출력
        elif hasattr(m, 'c1') and hasattr(m, 'c2'):
            print(f"Layer {i} [{layer_type}]: 요구 입력 채널(c1)={m.c1}, 출력 채널(c2)={m.c2}")
        else:
             print(f"Layer {i} [{layer_type}]: 상세 속성 생략")

Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     


  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  1    111872  ultralytics.nn.modules.block.C3k2            [128, 256, 1, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  4                  -1  1    444928  ultralytics.nn.modules.block.C3k2            [256, 512, 1, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  6                  -1  1   1380352  ultralytics.nn.modules.block.C3k2            [512, 512, 1, True]           
  7                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  8                  -1  1   1380352  ultralytics.nn.modules.block.C3k2            [512,

In [32]:
# [노트북 셀: 실제 텐서 Shape 확인 및 Bridge 인스턴스화]

import torch
import torch.nn as nn
import timm

# 1. 백본 생성 (dynamic_img_size 옵션 추가!)
swin_backbone = timm.create_model(
    'swin_tiny_patch4_window7_224', 
    pretrained=True, 
    features_only=True, 
    out_indices=(1, 2, 3),
    dynamic_img_size=True,  # ✅ 추가됨
    img_size=640            # ✅ 추가됨 (dummy_input 크기와 일치)
)

# 2. 더미 데이터 통과 (640x640)
dummy_input = torch.randn(1, 3, 640, 640)
with torch.no_grad():
    features = swin_backbone(dummy_input)

# 3. 실제 채널 확인
actual_channels = []
for i, f in enumerate(features):
    ch = f.shape[1] 
    actual_channels.append(ch)
    print(f"Feature {i} shape: {f.shape} -> Channel: {ch}")

print(f"\n✅ 추출된 actual_channels: {actual_channels}")

Feature 0 shape: torch.Size([1, 80, 80, 192]) -> Channel: 80
Feature 1 shape: torch.Size([1, 40, 40, 384]) -> Channel: 40
Feature 2 shape: torch.Size([1, 20, 20, 768]) -> Channel: 20

✅ 추출된 actual_channels: [80, 40, 20]


In [33]:
# 1. 브릿지 클래스 다시 정의 (텐서 순서 permute 포함)
class FeatureAlignmentBridge(nn.Module):
    def __init__(self, swin_channels, target_neck_channels):
        super().__init__()
        # Swin의 출력(192, 384, 768)을 YOLO Neck이 기대하는 입력으로 강제 변환
        self.bridge_p3 = nn.Conv2d(swin_channels[0], target_neck_channels[0], kernel_size=1)
        self.bridge_p4 = nn.Conv2d(swin_channels[1], target_neck_channels[1], kernel_size=1)
        self.bridge_p5 = nn.Conv2d(swin_channels[2], target_neck_channels[2], kernel_size=1)
        self.act = nn.SiLU(inplace=True)

    def forward(self, features):
        # features: [Batch, Channel, H, W] 형태로 들어옴
        p3 = self.act(self.bridge_p3(features[0]))
        p4 = self.act(self.bridge_p4(features[1]))
        p5 = self.act(self.bridge_p5(features[2]))
        return [p3, p4, p5]

# 2. YOLO Neck의 요구사항에 맞춘 타겟 채널 설정
# (에러 메시지 "expected input ... to have 1024 channels"를 반영하여 P5 매핑을 1024로 상향 조정)
target_channels = [256, 512, 1024] 

# 3. 모델에 브릿지 장착
model.bridge = FeatureAlignmentBridge(actual_channels, target_channels)

# 4. _predict_once 메서드 오버라이딩 (인덱싱 오류 수정본)
def patched_predict_once(self, x, profile=False, visualize=False, embed=None):
    if not hasattr(self, 'swin_backbone'):
         return [] # 초기화 시 빈 리스트 반환 방지
    
    features = self.swin_backbone(x)
    aligned_features = self.bridge(features)

    y = [None] * len(self.model) 
    
    # YOLO26m 구조상 순정 백본의 P3, P4, P5 위치에 텐서 주입
    y[4] = aligned_features[0]  
    y[6] = aligned_features[1]  
    y[9] = aligned_features[2]  

    x = aligned_features[2]

    for i, m in enumerate(self.model):
        if i >= 10: 
            if m.f != -1:
                if isinstance(m.f, int):
                    x = y[m.f] if m.f != -1 else x
                else:
                    x = [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y[i] = x if m.i in self.save else None
    return x

import types
model._predict_once = types.MethodType(patched_predict_once, model)

# 5. 대망의 통합 Forward Pass 테스트
print("\n🚀 최종 Forward Pass 테스트 시작...")
try:
    final_output = model(dummy_input)
    print("✅ Forward Pass 성공! 텐서 차원이 완벽하게 일치합니다.")
    # 최종 출력 텐서의 형태 확인 (감지된 바운딩 박스 정보 등)
    if isinstance(final_output, tuple):
        print(f"최종 출력 형태 (Tuple): {[o.shape for o in final_output]}")
    else:
        print(f"최종 출력 형태 (Tensor): {final_output.shape}")
        
except Exception as e:
    print(f"❌ 에러 발생: {e}")


🚀 최종 Forward Pass 테스트 시작...
❌ 에러 발생: Given groups=1, weight of size [512, 512, 1, 1], expected input[1, 1024, 20, 768] to have 512 channels, but got 1024 channels instead


💻 [노트북 셀: 순정 YOLO26m 채널 분석]

In [34]:
import torch
from ultralytics.nn.tasks import DetectionModel

print("--- 🔍 순정 YOLO26m 채널 분석 ---")

# 1. 뼈대(Swin 주입 전)가 되는 순정 YOLO 모델 인스턴스화
# (이때는 순정 CNN 백본이 들어가 있습니다)
pure_yolo = DetectionModel("yolo26m.yaml", nc=7)
pure_yolo.eval()

# 2. 더미 데이터 통과시키며 각 레이어의 출력 채널 기록
dummy_input = torch.randn(1, 3, 640, 640)
yolo_outputs = []

with torch.no_grad():
    x = dummy_input
    for i, m in enumerate(pure_yolo.model):
        if m.f != -1:
            if isinstance(m.f, int):
                x = yolo_outputs[m.f]
            else:
                x = [x if j == -1 else yolo_outputs[j] for j in m.f]
        
        x = m(x)
        yolo_outputs.append(x)
        
        # 우리가 Swin 결과물로 대체하려는 4번, 6번, 9번 레이어의 형태 확인
        if i in [4, 6, 9]:
            print(f"순정 YOLO Layer {i} 출력 Shape: {x.shape} -> 채널 수: {x.shape[1]}")

print("\n💡 이 채널 수들이 바로 브릿지가 출력해야 할 'target_channels' 입니다!")

--- 🔍 순정 YOLO26m 채널 분석 ---
Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  1    111872  ultralytics.nn.modules.block.C3k2            [128, 256, 1, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  4                  -1  1    444928  ultralytics.nn.modules.block.C3k2            [256, 512, 1, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  6                  -1  1   1380352  ultralytics.nn.modules.block.C3k2            [512, 512, 1, True]           
  7                  -

In [37]:
# [노트북 셀: 브릿지 채널 조정 및 최종 Forward Pass 검증]
import torch.nn as nn
import types

class FeatureAlignmentBridge(nn.Module):
    def __init__(self, swin_channels, target_neck_channels):
        super().__init__()
        # Swin의 출력(192, 384, 768)을 YOLO Neck이 기대하는 입력(512, 512, 512)으로 강제 변환
        self.bridge_p3 = nn.Conv2d(swin_channels[0], target_neck_channels[0], kernel_size=1)
        self.bridge_p4 = nn.Conv2d(swin_channels[1], target_neck_channels[1], kernel_size=1)
        self.bridge_p5 = nn.Conv2d(swin_channels[2], target_neck_channels[2], kernel_size=1)
        self.act = nn.SiLU(inplace=True)

    def forward(self, features):
        # 💡 핵심 수정: [Batch, H, W, Channel] -> [Batch, Channel, H, W] 로 순서 변경
        p3 = self.act(self.bridge_p3(features[0].permute(0, 3, 1, 2)))
        p4 = self.act(self.bridge_p4(features[1].permute(0, 3, 1, 2)))
        p5 = self.act(self.bridge_p5(features[2].permute(0, 3, 1, 2)))
        return [p3, p4, p5]

# 1. 타겟 채널 완벽 고정
target_channels = [512, 512, 512] 
model.bridge = FeatureAlignmentBridge([192, 384, 768], target_channels)

# 2. _predict_once 메서드 동적 재정의 (인덱싱 오류 수정본)
def patched_predict_once(self, x, profile=False, visualize=False, embed=None):
    if not hasattr(self, 'swin_backbone'):
        return []
        
    features = self.swin_backbone(x)
    aligned_features = self.bridge(features)

    y = [None] * len(self.model) 
    
    # YOLO26m 구조상 순정 백본의 P3, P4, P5 위치(4, 6, 9번)에 텐서 주입
    y[4] = aligned_features[0]  
    y[6] = aligned_features[1]  
    y[9] = aligned_features[2]  

    x = aligned_features[2]

    for i, m in enumerate(self.model):
        if i >= 10: 
            if m.f != -1:
                if isinstance(m.f, int):
                    x = y[m.f] if m.f != -1 else x
                else:
                    x = [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y[i] = x if m.i in self.save else None
    return x

model._predict_once = types.MethodType(patched_predict_once, model)

# 3. 최종 통합 테스트!
print("\n🚀 최종 Forward Pass 테스트 시작...")
try:
    final_output = model(dummy_input)
    print("✅ Forward Pass 성공! 텐서 차원이 완벽하게 일치합니다.")
    print("✅ 이제 이 로직을 models/swin_yolo26.py 에 반영하고 학습을 시작하세요!")
except Exception as e:
    print(f"❌ 에러 발생: {e}")


🚀 최종 Forward Pass 테스트 시작...
✅ Forward Pass 성공! 텐서 차원이 완벽하게 일치합니다.
✅ 이제 이 로직을 models/swin_yolo26.py 에 반영하고 학습을 시작하세요!
